# Tool Executor 完全指南

**前置知识**: Function Calling、Tool Registry

**学习目标**: 掌握安全执行工具的方法，包括超时控制、重试机制、执行历史追踪

---

## 核心问题：为什么需要执行器？

直接调用工具函数存在风险：
- **无限阻塞**: 网络请求卡住，整个 Agent 停止响应
- **异常崩溃**: 未捕获的异常导致程序终止
- **无法追踪**: 不知道执行了什么、花了多长时间
- **无法重试**: 临时失败（如网络抖动）直接报错

**Tool Executor 的解决方案**：
```
FunctionCall → [超时控制] → [错误捕获] → [重试机制] → [历史记录] → ExecutionResult
```

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import sys
import time

sys.path.insert(0, '..')

from src.tool_registry import ToolRegistry, tool
from src.tool_executor import (
    ToolExecutor,
    ExecutionResult,
    ExecutionContext,
    ExecutionStatus,
)
from src.function_calling import FunctionCall

print("导入成功！")

---

## 第一步：创建执行器

执行器需要一个 `ToolRegistry` 来查找工具。

In [ ]:
# ============================================================
# 创建注册表和执行器
# ============================================================

# 1. 创建注册表并注册工具
registry = ToolRegistry()

@registry.register(tags=["math"])
def add(a: int, b: int) -> int:
    """两数相加"""
    return a + b

@registry.register(tags=["math"])
def divide(a: int, b: int) -> float:
    """两数相除"""
    return a / b

@registry.register(tags=["slow"])
def slow_task(seconds: float) -> str:
    """模拟耗时任务"""
    time.sleep(seconds)
    return f"完成，耗时 {seconds} 秒"

# 2. 创建执行器
executor = ToolExecutor(
    registry=registry,
    default_timeout=5.0,  # 默认超时 5 秒
    max_retries=3,        # 默认最大重试 3 次
)

print(f"执行器已创建: {executor}")

---

## 第二步：基本执行

`execute()` 方法返回 `ExecutionResult`，包含执行状态、输出、耗时等信息。

In [ ]:
# ============================================================
# 基本执行
# ============================================================

# 成功执行
result = executor.execute("add", {"a": 10, "b": 20})

print(f"状态: {result.status.value}")
print(f"输出: {result.output}")
print(f"耗时: {result.execution_time:.4f} 秒")
print(f"是否成功: {result.is_success}")

In [ ]:
# ============================================================
# ExecutionResult 的便捷方法
# ============================================================

# to_message(): 生成适合返回给 LLM 的消息
print("LLM 消息:")
print(result.to_message())

print("\n序列化为字典:")
print(result.to_dict())

---

## 第三步：错误处理

执行器会捕获所有异常，不会让程序崩溃。

In [ ]:
# ============================================================
# 错误处理示例
# ============================================================

# 1. 除零错误
result = executor.execute("divide", {"a": 10, "b": 0})
print(f"除零错误:")
print(f"  状态: {result.status.value}")
print(f"  错误: {result.error.split(chr(10))[0]}...")  # 只显示第一行

# 2. 工具不存在
result = executor.execute("nonexistent", {})
print(f"\n工具不存在:")
print(f"  状态: {result.status.value}")
print(f"  错误: {result.error}")

# 3. 工具被禁用
registry.disable("add")
result = executor.execute("add", {"a": 1, "b": 2})
print(f"\n工具被禁用:")
print(f"  状态: {result.status.value}")
print(f"  错误: {result.error}")
registry.enable("add")  # 恢复

---

## 第四步：超时控制

防止工具执行时间过长阻塞整个系统。

In [ ]:
# ============================================================
# 超时控制
# ============================================================

# 正常执行（在超时内完成）
result = executor.execute("slow_task", {"seconds": 0.5}, timeout=2.0)
print(f"正常完成: {result.status.value}, 输出: {result.output}")

# 超时（任务需要 10 秒，但只给 0.5 秒）
result = executor.execute("slow_task", {"seconds": 10.0}, timeout=0.5)
print(f"\n超时: {result.status.value}")
print(f"错误信息: {result.error}")

---

## 第五步：执行 FunctionCall

直接执行从 LLM 解析出的 `FunctionCall` 对象。

In [ ]:
# ============================================================
# 执行 FunctionCall
# ============================================================

# 模拟从 LLM 解析出的调用
call = FunctionCall(
    name="add",
    arguments={"a": 100, "b": 200},
    id="call_abc123",  # 可选的调用 ID
)

result = executor.execute_call(call)

print(f"调用 ID: {result.call_id}")
print(f"输出: {result.output}")

---

## 第六步：批量执行

一次执行多个函数调用，支持顺序和并行两种模式。

In [ ]:
# ============================================================
# 批量执行
# ============================================================

calls = [
    FunctionCall(name="add", arguments={"a": 1, "b": 1}),
    FunctionCall(name="add", arguments={"a": 2, "b": 2}),
    FunctionCall(name="add", arguments={"a": 3, "b": 3}),
]

# 顺序执行
start = time.time()
results = executor.execute_batch(calls, parallel=False)
print(f"顺序执行耗时: {time.time() - start:.3f}s")
print(f"结果: {[r.output for r in results]}")

# 并行执行
start = time.time()
results = executor.execute_batch(calls, parallel=True)
print(f"\n并行执行耗时: {time.time() - start:.3f}s")
print(f"结果: {[r.output for r in results]}")

---

## 第七步：重试机制

对于可能临时失败的操作（如网络请求），自动重试。

In [ ]:
# ============================================================
# 重试机制
# ============================================================

# 创建一个会失败几次然后成功的工具
attempt_count = 0

@registry.register(tags=["flaky"])
def flaky_task() -> str:
    """模拟不稳定的任务（前2次失败，第3次成功）"""
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        raise ConnectionError(f"第 {attempt_count} 次尝试失败")
    return f"第 {attempt_count} 次尝试成功！"

# 重置计数器
attempt_count = 0

# 使用重试执行
result = executor.execute_with_retry(
    "flaky_task",
    {},
    max_retries=5,
    retry_delay=0.1,  # 重试间隔 0.1 秒
)

print(f"状态: {result.status.value}")
print(f"输出: {result.output}")

---

## 第八步：执行上下文

`ExecutionContext` 携带用户信息、权限、共享变量等。

In [ ]:
# ============================================================
# 执行上下文
# ============================================================

context = ExecutionContext(
    user_id="user_123",
    session_id="session_456",
    permissions=["read", "write"],
    timeout=10.0,
    variables={"api_key": "xxx"},
)

# 权限检查
print(f"有 read 权限: {context.has_permission('read')}")
print(f"有 delete 权限: {context.has_permission('delete')}")

# 变量访问
print(f"\nAPI Key: {context.get_variable('api_key')}")
print(f"不存在的变量: {context.get_variable('missing', 'default')}")

# 通配符权限
admin_context = ExecutionContext(permissions=["*"])
print(f"\n管理员有任意权限: {admin_context.has_permission('anything')}")

---

## 第九步：生命周期钩子

在执行前后插入自定义逻辑（日志、监控、审计等）。

In [ ]:
# ============================================================
# 生命周期钩子
# ============================================================

# 定义钩子函数
def log_before(tool, args, ctx):
    print(f"[BEFORE] 即将执行: {tool.name}({args})")

def log_after(tool, args, ctx, result):
    print(f"[AFTER] 执行完成: {tool.name} -> {result.status.value}")

def log_error(tool, args, ctx, result):
    print(f"[ERROR] 执行失败: {tool.name}, 错误: {result.error.split(chr(10))[0]}")

# 注册钩子
executor.add_hook("before_execute", log_before)
executor.add_hook("after_execute", log_after)
executor.add_hook("on_error", log_error)

# 测试
print("=== 成功执行 ===")
executor.execute("add", {"a": 1, "b": 2})

print("\n=== 失败执行 ===")
executor.execute("divide", {"a": 1, "b": 0})

# 移除钩子
executor.remove_hook("before_execute", log_before)
executor.remove_hook("after_execute", log_after)
executor.remove_hook("on_error", log_error)

---

## 第十步：执行历史与统计

追踪所有执行记录，分析性能和成功率。

In [ ]:
# ============================================================
# 执行历史
# ============================================================

# 清空历史
executor.clear_history()

# 执行一些操作
executor.execute("add", {"a": 1, "b": 1})
executor.execute("add", {"a": 2, "b": 2})
executor.execute("divide", {"a": 10, "b": 2})
executor.execute("divide", {"a": 1, "b": 0})  # 会失败

# 获取历史
history = executor.get_history()
print(f"总执行次数: {len(history)}")

# 按状态过滤
success_history = executor.get_history(status=ExecutionStatus.SUCCESS)
print(f"成功次数: {len(success_history)}")

# 按工具过滤
add_history = executor.get_history(tool_name="add")
print(f"add 工具执行次数: {len(add_history)}")

In [ ]:
# ============================================================
# 执行统计
# ============================================================

stats = executor.get_stats()

print("执行统计:")
print(f"  总执行次数: {stats['total_executions']}")
print(f"  成功次数: {stats['success_count']}")
print(f"  失败次数: {stats['error_count']}")
print(f"  成功率: {stats['success_rate']:.1%}")
print(f"  平均耗时: {stats['average_time']:.4f} 秒")
print(f"  总耗时: {stats['total_time']:.4f} 秒")

---

## 实战：完整的 Agent 执行流程

In [ ]:
# ============================================================
# 完整示例：Agent 工具执行流程
# ============================================================
from src.function_calling import FunctionCallParser

# 1. 设置
agent_registry = ToolRegistry()

@agent_registry.register(tags=["math"])
def calculator(expression: str) -> float:
    """计算数学表达式"""
    return eval(expression)

@agent_registry.register(tags=["text"])
def word_count(text: str) -> int:
    """统计文本字数"""
    return len(text)

agent_executor = ToolExecutor(agent_registry, default_timeout=5.0)

# 2. 创建解析器
functions = [t.to_function_definition() for t in agent_registry.list_tools()]
parser = FunctionCallParser(functions)

# 3. 模拟 LLM 输出
llm_outputs = [
    '{"name": "calculator", "arguments": {"expression": "2 ** 10"}}',
    '{"name": "word_count", "arguments": {"text": "Hello World"}}',
    '{"name": "calculator", "arguments": {"expression": "1/0"}}',  # 会失败
]

# 4. 处理流程
print("=== Agent 执行流程 ===")
for i, output in enumerate(llm_outputs, 1):
    print(f"\n--- 调用 {i} ---")
    
    # 解析
    calls = parser.parse(output)
    if not calls:
        print("未检测到函数调用")
        continue
    
    call = calls[0]
    print(f"解析: {call.name}({call.arguments})")
    
    # 验证
    errors = parser.validate(call)
    if errors:
        print(f"验证失败: {errors}")
        continue
    
    # 执行
    result = agent_executor.execute_call(call)
    print(f"结果: {result.to_message()}")

---

## 本节要点

| 概念 | 作用 | 关键方法 |
|------|------|----------|
| `ToolExecutor` | 安全执行工具 | `execute()`, `execute_call()`, `execute_batch()` |
| `ExecutionResult` | 执行结果 | `is_success`, `to_message()`, `to_dict()` |
| `ExecutionContext` | 执行上下文 | `has_permission()`, `get_variable()` |
| `ExecutionStatus` | 执行状态枚举 | `SUCCESS`, `ERROR`, `TIMEOUT` |

**核心功能**:
- 超时控制: `execute(..., timeout=5.0)`
- 重试机制: `execute_with_retry(..., max_retries=3)`
- 生命周期钩子: `add_hook("before_execute", callback)`
- 执行历史: `get_history()`, `get_stats()`

**下一步**: 学习如何将所有组件集成，构建完整的 Agent 系统。